In [0]:
pip install azure-keyvault-secrets

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
pip install supabase

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
pip install --upgrade typing_extensions supabase

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
pip install --upgrade websockets supabase realtime

  Using cached websockets-17.0.1-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (6.3 kB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:

# import Azure SDK library
from azure.keyvault.secrets import SecretClient 
import os
from supabase import create_client, Client
import pandas as pd
import json
import os
import hashlib


from pyspark.sql.functions import *
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StringType, StructType, IntegerType, BooleanType, VarcharType,TimestampType,FloatType
from pyspark.sql.functions import year
from pyspark.sql import SparkSession

# service credentials from Unity Catalog
credential = dbutils.credentials.getServiceCredentialsProvider('key-vault-access-connector')
# Azure Key Vault url - copy it from vault properties
vault_url = "https://traveljournalkeyvault.vault.azure.net/"
# init client
client = SecretClient(vault_url=vault_url, credential=credential)

spark = SparkSession.builder.appName("test").getOrCreate()

supabase_url= client.get_secret("supabase-url").value
supabase_key= client.get_secret("supabase-key").value

supabase_url_str = str(supabase_url)
supabase_key_str = str(supabase_key)

source_path = "abfss://travelsource@databricktraveljournal.dfs.core.windows.net"


supabase: Client = create_client(supabase_url_str, supabase_key_str)

table_list = [
    "accounts",
    "activity_code",
    "activity_tag",
    "bio",
    "country_code",
    "follows",
    "google_maps_address",
    "images",
    "location_country_tag",
    "plan_trip",
    "post_bookmarks",
    "post_likes",
    "trip_posts",
    "trip_stops"
]

df_schema = StructType([
    StructField('id', IntegerType(), False),
    StructField('user_id', IntegerType(), True),
    StructField('image_id', IntegerType(), True),
    StructField('trip_name', StringType(), True),
    StructField('location_country_tag_id', IntegerType(), True),
    StructField('activity_type_tag_id', IntegerType(), True),
    StructField('total_distance', IntegerType(), True),
    StructField('caption', StringType(), True),
    StructField('created_at', StringType(), True),
    StructField('flag', BooleanType(), True),
    StructField('trip_rating', FloatType(), True)


])

def data_partitioned(df,table):
    
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

    try:
        # Enable dynamic partition overwrite
        path = f"{source_path}/{table}"
        # Write data partitioned by year and month
        df.write.partitionBy("year", "month","day").mode("overwrite").format("parquet").save(path)
    
    except Exception as e:
            print("==============")
            print(f"Error: {e}")
            print("==============")

def extract_date(df):
    N = 10  # last 7 days
    '''
    df_current_date_filter = df.filter(
        col("date_type").between(date_sub(current_date(), N), current_date())
    )
    '''
    #df_current_date_filter = df.filter(col("date_type") == current_date()-1 )
    #df_current_date_filter = df.filter(col("date_type") == "2024-08-06")
    #df_current_date_filter = df.filter(col("year") == "4")
    
    try:
        extract_date_df = (
            df.withColumn("year", year("date_type"))
            .withColumn("month", month("date_type"))
            .withColumn("day", day("date_type"))
            )
        extract_date_df = df.filter(col("year") == 2024)   # or 2026, whichever you want

        
        extract_date_df.show()
    except Exception as e:
            print("==============")
            print(f"Error: {e}")
            print("==============")

    return extract_date_df


def spark_change_data_type(df):
    try:
        if 'created_at' in df.columns:
            df = df.withColumn("created_at", to_utc_timestamp(df["created_at"], "Asia/Tokyo"))

            df = df.withColumn("date_type",to_date("created_at"))
            
            print(df.show())
        elif 'upload_at' in df.columns:
            df = df.withColumn("date_type",to_date("upload_at"))
            print(df.show())
            
    except Exception as e:
            print("==============")
            print(f"Error: {e}")
            print("==============")

    return df

def spark_create_dataframe(table_list):
    for table in table_list:
        try:
            response = (
                supabase.table(table)
                .select("*")
                .execute())
            data = response.model_dump_json()
            json_object = json.loads(data)
            json_data = json_object["data"]
        except Exception as e:
            print(f"Error: {e}")

        try:
            if table == "trip_posts":
                df = spark.createDataFrame(json_data,schema = df_schema)
                print("===============")
                print(f"Table: {table}")
                print(type(json_data))
                print("===============")

                df = spark_change_data_type(df)
                extract_date_df = extract_date(df)
                data_partitioned(extract_date_df,table)

            else:
                df = spark.createDataFrame(json_data)
                print("===============")
                print(f"Table: {table}")
                print(type(json_data))
                print("===============")

                df = spark_change_data_type(df)
                extract_date_df = extract_date(df)
                data_partitioned(extract_date_df,table)

                
        except Exception as e:
            print("==============")
            print(f"Error at {table}")
            print(f"Error: {e}")
            print("==============")




if __name__ == "__main__":
    spark_create_dataframe(table_list)

Table: accounts
<class 'list'>
+--------------------+--------------------+-----+--------+--------------------+--------------------+-----------+----+-------+--------------+----------+
|          created_at|               email| flag|image_id|          last_login|        passwordhash|preferences|role|user_id|      username| date_type|
+--------------------+--------------------+-----+--------+--------------------+--------------------+-----------+----+-------+--------------+----------+
|2026-06-06 19:37:...|      test@gmail.com|false|    NULL|2026-07-17T06:59:...|$2b$10$7X0xq5PEP0...|       test|user|      6|          test|2026-06-06|
|2026-06-16 00:08:...|  traveler@gmail.com|false|    NULL|2026-06-16T09:09:...|$2b$10$dAnskLXtf2...|      test2|user|      7|         test2|2026-06-16|
|2026-06-17 22:39:...|     david@gmail.com|false|    NULL|                NULL|$2b$10$u924ea4.Xu...|      david|user|      8|         david|2026-06-17|
|2026-07-02 05:32:...|  gpratt@example.org|false|     508

{"ts": "2026-08-12 03:50:47.838", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`email`, `flag`, `role`, `user_id`, `username`]. SQLSTATE: 42703", "context": {"file": "<command-7457877521212424>, line 103 in cell [1]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o533.filter.\n: org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`email`, `flag`, `role`, `user_id`, `username`]. SQLSTATE: 42703;\n'Filter '`=`('year, 2024)\n+- Project [created_at#21940, email#21921, flag#21922, image_id#21923L, last_login#21924, passwordhash#21925, preferences#21926,

Error: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`email`, `flag`, `role`, `user_id`, `username`]. SQLSTATE: 42703;
'Filter '`=`('year, 2024)
+- Project [created_at#21940, email#21921, flag#21922, image_id#21923L, last_login#21924, passwordhash#21925, preferences#21926, role#21927, user_id#21928L, username#21929, to_date(created_at#21940, None, Some(Etc/UTC), false) AS date_type#21951]
   +- Project [to_utc_timestamp(cast(created_at#21920 as timestamp), Asia/Tokyo) AS created_at#21940, email#21921, flag#21922, image_id#21923L, last_login#21924, passwordhash#21925, preferences#21926, role#21927, user_id#21928L, username#21929]
      +- LogicalRDD [created_at#21920, email#21921, flag#21922, image_id#21923L, last_login#21924, passwordhash#21925, preferences#21926, role#21927, user_id#21928L, username#21929], false

Table: activity_code
<class 'list'>
+-------------+----------------+

{"ts": "2026-08-12 03:50:55.565", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `flag`, `emoji`, `created_at`, `date_type`]. SQLSTATE: 42703", "context": {"file": "<command-7457877521212424>, line 103 in cell [1]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o598.filter.\n: org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `flag`, `emoji`, `created_at`, `date_type`]. SQLSTATE: 42703;\n'Filter '`=`('year, 2024)\n+- Project [activity_code#22079L, activity_name#22080, created_at#22091, emoji#22082, flag#22083, id#22084L, to_date(created

Error: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `flag`, `emoji`, `created_at`, `date_type`]. SQLSTATE: 42703;
'Filter '`=`('year, 2024)
+- Project [activity_code#22079L, activity_name#22080, created_at#22091, emoji#22082, flag#22083, id#22084L, to_date(created_at#22091, None, Some(Etc/UTC), false) AS date_type#22098]
   +- Project [activity_code#22079L, activity_name#22080, to_utc_timestamp(cast(created_at#22081 as timestamp), Asia/Tokyo) AS created_at#22091, emoji#22082, flag#22083, id#22084L]
      +- LogicalRDD [activity_code#22079L, activity_name#22080, created_at#22081, emoji#22082, flag#22083, id#22084L], false

Table: activity_tag
<class 'list'>
+-------------+--------------------+-----+---+------------+----------+
|activity_code|          created_at| flag| id|trip_post_id| date_type|
+-------------+--------------------+-----+---+------------+----------+
|         

...

Error: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `user_id`, `post_id`, `created_at`, `date_type`]. SQLSTATE: 42703;
'Filter '`=`('year, 2024)
+- Project [created_at#23239, id#23232L, post_id#23233L, user_id#23234L, to_date(created_at#23239, None, Some(Etc/UTC), false) AS date_type#23244]
   +- Project [to_utc_timestamp(cast(created_at#23231 as timestamp), Asia/Tokyo) AS created_at#23239, id#23232L, post_id#23233L, user_id#23234L]
      +- LogicalRDD [created_at#23231, id#23232L, post_id#23233L, user_id#23234L], false

Table: trip_posts
<class 'list'>
+---+-------+--------+--------------------+-----------------------+--------------------+--------------+--------------------+--------------------+-----+-----------+----------+
| id|user_id|image_id|           trip_name|location_country_tag_id|activity_type_tag_id|total_distance|             caption|          created_at| flag|tr

{"ts": "2026-08-12 03:51:57.427", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `flag`, `caption`, `user_id`, `image_id`]. SQLSTATE: 42703", "context": {"file": "<command-7457877521212424>, line 103 in cell [1]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1226.filter.\n: org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `flag`, `caption`, `user_id`, `image_id`]. SQLSTATE: 42703;\n'Filter '`=`('year, 2024)\n+- Project [id#23318, user_id#23319, image_id#23320, trip_name#23321, location_country_tag_id#23322, activity_type_tag_id#23323

Error: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `flag`, `caption`, `user_id`, `image_id`]. SQLSTATE: 42703;
'Filter '`=`('year, 2024)
+- Project [id#23318, user_id#23319, image_id#23320, trip_name#23321, location_country_tag_id#23322, activity_type_tag_id#23323, total_distance#23324, caption#23325, created_at#23340, flag#23327, trip_rating#23328, to_date(created_at#23340, None, Some(Etc/UTC), false) AS date_type#23352]
   +- Project [id#23318, user_id#23319, image_id#23320, trip_name#23321, location_country_tag_id#23322, activity_type_tag_id#23323, total_distance#23324, caption#23325, to_utc_timestamp(cast(created_at#23326 as timestamp), Asia/Tokyo) AS created_at#23340, flag#23327, trip_rating#23328]
      +- LogicalRDD [id#23318, user_id#23319, image_id#23320, trip_name#23321, location_country_tag_id#23322, activity_type_tag_id#23323, total_distance#23324, caption#23325,

{"ts": "2026-08-12 03:52:06.301", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `lat`, `flag`, `lng`, `hours`]. SQLSTATE: 42703", "context": {"file": "<command-7457877521212424>, line 103 in cell [1]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1284.filter.\n: org.apache.spark.sql.catalyst.ExtendedAnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `lat`, `flag`, `lng`, `hours`]. SQLSTATE: 42703;\n'Filter '`=`('year, 2024)\n+- Project [address#23489, category#23490, created_at#23527, duration#23492, flag#23493, google_maps_address_id#23494L, hours#23495, id#23496L, image

Error: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `year` cannot be resolved. Did you mean one of the following? [`id`, `lat`, `flag`, `lng`, `hours`]. SQLSTATE: 42703;
'Filter '`=`('year, 2024)
+- Project [address#23489, category#23490, created_at#23527, duration#23492, flag#23493, google_maps_address_id#23494L, hours#23495, id#23496L, image_id#23497L, lat#23498, lng#23499, location_title#23500, price_range#23501, rating#23502, review_count#23503L, status#23504, time#23505, trip_post_id#23506L, user_id#23507L, to_date(created_at#23527, None, Some(Etc/UTC), false) AS date_type#23547]
   +- Project [address#23489, category#23490, to_utc_timestamp(cast(created_at#23491 as timestamp), Asia/Tokyo) AS created_at#23527, duration#23492, flag#23493, google_maps_address_id#23494L, hours#23495, id#23496L, image_id#23497L, lat#23498, lng#23499, location_title#23500, price_range#23501, rating#23502, review_count#23503L, status#23504, time#23505, trip_pos